# Formative User Feedback Round 2 analysis

This notebook loads the three Round 2 export ZIPs and summarizes task performance, Adaptive Music decisions, and transitions.

**Scope:** formative evidence from three Adaptive Music sessions. It describes patterns and cannot establish that music caused a performance change.

In [2]:
from pathlib import Path
from io import BytesIO
import json, zipfile
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

# One outer ZIP containing the three participant-results ZIP files.
ITERATION_ZIP = Path("/Users/zi/Documents/GitHub/TunedIn-Web/feedback-files/results/iterative-2.zip")
if not ITERATION_ZIP.exists():
    raise FileNotFoundError(f"Iteration ZIP not found: {ITERATION_ZIP}")

## 1. Load the iteration archive

Set `ITERATION_ZIP` above to one outer archive. That archive must contain the individual participant result ZIP files. The loader ignores macOS metadata files and accepts exports stored either at the inner ZIP root or inside one folder.

In [3]:
def archive_member(names, filename):
    matches = [name for name in names if name == filename or name.endswith("/" + filename)]
    if not matches:
        raise KeyError(f"{filename} was not found in this participant export ZIP.")
    return matches[0]

def load_participant_export(zip_bytes, label):
    with zipfile.ZipFile(BytesIO(zip_bytes)) as archive:
        names = [name for name in archive.namelist() if not name.startswith("__MACOSX/")]
        def read_csv(filename):
            return pd.read_csv(BytesIO(archive.read(archive_member(names, filename))))
        return {
            "label": label,
            "session": json.loads(archive.read(archive_member(names, "session.json"))),
            "blocks": read_csv("blocks.csv"),
            "records": read_csv("records.csv"),
            "decisions": read_csv("decisions.csv"),
            "transitions": read_csv("transitions.csv"),
        }

with zipfile.ZipFile(ITERATION_ZIP) as iteration_archive:
    inner_zips = [
        name for name in iteration_archive.namelist()
        if name.lower().endswith(".zip") and not name.startswith("__MACOSX/")
    ]
    if not inner_zips:
        raise ValueError("No participant ZIP files were found inside the iteration archive.")
    bundles = [
        load_participant_export(iteration_archive.read(name), name)
        for name in inner_zips
    ]

sessions = pd.DataFrame([bundle["session"] for bundle in bundles])
blocks = pd.concat([bundle["blocks"] for bundle in bundles], ignore_index=True)
records = pd.concat([bundle["records"] for bundle in bundles], ignore_index=True)
decisions = pd.concat([bundle["decisions"] for bundle in bundles], ignore_index=True)
transitions = pd.concat([bundle["transitions"] for bundle in bundles], ignore_index=True)

print(f"Participant export ZIPs found: {len(bundles)}")
print(f"Participants: {sessions.participantId.nunique()}")
print(f"Block rows: {len(blocks)} | Record rows: {len(records)}")
print(f"Adaptive decisions: {len(decisions)} | Transitions: {len(transitions)}")
sessions[["participantId", "studyMode", "buildVersion", "randomizationSeed",
          "comfortVolumePercent", "timing", "conditionOrder", "endedEarlyAt"]]

Participant export ZIPs found: 3
Participants: 3
Block rows: 9 | Record rows: 91
Adaptive decisions: 27 | Transitions: 13


,participantId,studyMode,buildVersion,randomizationSeed,comfortVolumePercent,timing,conditionOrder,endedEarlyAt
0,F2-P003,Formative,fuf-iteration-2.0.1,fuf2.0.1,75,"{'practiceSeconds': 60, 'baselineSeconds': 120...",[Adaptive music],None
1,P001,Formative,fuf-iteration-2.0.1-timing-1m-2m-5m,fuf2.0.1,59,"{'practiceSeconds': 60, 'baselineSeconds': 120...",[Adaptive music],None
2,F2-P002,Formative,fuf-iteration-2.0.1,fuf2.0.1,60,"{'practiceSeconds': 60, 'baselineSeconds': 120...",[Adaptive music],None


## 2. Task performance

Empty `interrupted_restart` rows are retained in the exports but excluded from performance summaries.

In [4]:
completed = blocks.loc[blocks["attempt_status"].eq("completed")].copy()
for column in ["block", "duration_seconds", "records_presented", "validated_records",
               "validated_records_per_minute", "median_il_ms", "median_fped_ms",
               "first_pass_error_rate", "correction_cycles"]:
    completed[column] = pd.to_numeric(completed[column], errors="coerce")

cols = ["participant_id", "block", "condition", "duration_seconds", "records_presented",
        "validated_records", "validated_records_per_minute", "median_il_ms",
        "median_fped_ms", "first_pass_error_rate", "correction_cycles"]
display(completed[cols].sort_values(["participant_id", "block"]))

practice = completed.loc[completed.block.eq(0)].copy()
baseline = completed.loc[completed.block.eq(1)].copy()
adaptive = completed.loc[completed.condition.eq("Adaptive music")].copy()

comparison = baseline.merge(adaptive, on="participant_id", suffixes=("_baseline", "_adaptive"))
comparison["throughput_change_pct"] = 100 * (
    comparison.validated_records_per_minute_adaptive /
    comparison.validated_records_per_minute_baseline - 1
)
comparison["il_change_pct"] = 100 * (
    comparison.median_il_ms_adaptive / comparison.median_il_ms_baseline - 1
)
comparison["fped_change_pct"] = 100 * (
    comparison.median_fped_ms_adaptive / comparison.median_fped_ms_baseline - 1
)
display(comparison[[
    "participant_id", "validated_records_baseline", "validated_records_per_minute_baseline",
    "validated_records_adaptive", "validated_records_per_minute_adaptive",
    "throughput_change_pct", "il_change_pct", "fped_change_pct",
    "first_pass_error_rate_adaptive",
]].round(2))

def weighted_throughput(frame):
    return frame.validated_records.sum() / (frame.duration_seconds.sum() / 60)

summary = pd.DataFrame({
    "stage": ["Practice", "Silent Baseline", "Adaptive block"],
    "validated_records": [practice.validated_records.sum(), baseline.validated_records.sum(), adaptive.validated_records.sum()],
    "weighted_validated_records_per_minute": [weighted_throughput(practice), weighted_throughput(baseline), weighted_throughput(adaptive)],
    "weighted_first_pass_error_rate": [
        (practice.first_pass_error_rate * practice.validated_records).sum() / practice.validated_records.sum(),
        (baseline.first_pass_error_rate * baseline.validated_records).sum() / baseline.validated_records.sum(),
        (adaptive.first_pass_error_rate * adaptive.validated_records).sum() / adaptive.validated_records.sum(),
    ],
})
summary.round(3)

,participant_id,block,condition,duration_seconds,records_presented,validated_records,validated_records_per_minute,median_il_ms,median_fped_ms,first_pass_error_rate,correction_cycles
6,F2-P002,0,No music,60,3,2,2.000,"2,268.850","14,837.600",0.500,1
7,F2-P002,1,No music,120,9,8,4.000,"2,018.050","12,341.900",0.000,0
8,F2-P002,2,Adaptive music,300,20,19,3.800,"1,583.600","11,538.800",0.158,3
0,F2-P003,0,No music,60,2,1,1.000,"3,484.500","25,498.700",0.000,0
1,F2-P003,1,No music,120,5,4,2.000,"2,786.850","25,041.500",0.000,0
2,F2-P003,2,Adaptive music,300,13,12,2.400,"2,797.800","18,962.000",0.083,1
3,P001,0,No music,60,1,0,0.000,"1,844.000","8,988.000",1.000,6
4,P001,1,No music,120,10,9,4.500,"1,272.000","10,600.000",0.000,0
5,P001,2,Adaptive music,300,28,27,5.400,"1,199.000","8,941.000",0.074,2


,participant_id,validated_records_baseline,validated_records_per_minute_baseline,validated_records_adaptive,validated_records_per_minute_adaptive,throughput_change_pct,il_change_pct,fped_change_pct,first_pass_error_rate_adaptive
0,F2-P003,4,2.000,12,2.400,20.000,0.390,-24.280,0.080
1,P001,9,4.500,27,5.400,20.000,-5.740,-15.650,0.070
2,F2-P002,8,4.000,19,3.800,-5.000,-21.530,-6.510,0.160


,stage,validated_records,weighted_validated_records_per_minute,weighted_first_pass_error_rate
0,Practice,3,1.000,0.333
1,Silent Baseline,21,3.500,0.000
2,Adaptive block,58,3.867,0.103


## 3. Adaptive-rule eligibility and triggers

These values match the Round 2 rule: at least 3 records in a 60-second window; Reduced at IL or FPED ≥120% of baseline or error >15%; Elevated at IL and FPED ≤95% of baseline and error ≤5%.

In [5]:
MINIMUM_RECORD_COUNT = 3
REDUCED_IL_MULTIPLIER, REDUCED_FPED_MULTIPLIER, REDUCED_ERROR_RATE = 1.20, 1.20, 0.15
ELEVATED_IL_MULTIPLIER, ELEVATED_FPED_MULTIPLIER, ELEVATED_ERROR_RATE = 0.95, 0.95, 0.05

for column in ["record_count", "median_il_ms", "median_fped_ms", "first_pass_error_rate",
               "baseline_il_ms", "baseline_fped_ms"]:
    decisions[column] = pd.to_numeric(decisions[column], errors="coerce")

decisions["eligible"] = decisions.record_count >= MINIMUM_RECORD_COUNT
decisions["il_ratio"] = decisions.median_il_ms / decisions.baseline_il_ms
decisions["fped_ratio"] = decisions.median_fped_ms / decisions.baseline_fped_ms
decisions["retained_while_waiting"] = decisions.reason.str.contains("waiting", case=False, na=False)

eligibility = decisions.groupby("participant_id", as_index=False).agg(
    evaluations=("eligible", "size"),
    eligible_evaluations=("eligible", "sum"),
    retained_while_waiting=("retained_while_waiting", "sum"),
    mean_records_in_window=("record_count", "mean"),
)
eligibility["eligible_percent"] = 100 * eligibility.eligible_evaluations / eligibility.evaluations
display(eligibility.round(2))

state_counts = decisions.pivot_table(
    index="participant_id", columns="selected_state", values="block",
    aggfunc="size", fill_value=0
).reset_index()
display(state_counts)

actual = decisions.loc[~decisions.retained_while_waiting].copy()
actual["reduced_il_trigger"] = actual.il_ratio >= REDUCED_IL_MULTIPLIER
actual["reduced_fped_trigger"] = actual.fped_ratio >= REDUCED_FPED_MULTIPLIER
actual["reduced_error_trigger"] = actual.first_pass_error_rate > REDUCED_ERROR_RATE
actual["elevated_all_conditions"] = (
    (actual.il_ratio <= ELEVATED_IL_MULTIPLIER)
    & (actual.fped_ratio <= ELEVATED_FPED_MULTIPLIER)
    & (actual.first_pass_error_rate <= ELEVATED_ERROR_RATE)
)
actual[[
    "participant_id", "record_count", "il_ratio", "fped_ratio",
    "first_pass_error_rate", "previous_state", "selected_state", "reason",
    "reduced_il_trigger", "reduced_fped_trigger", "reduced_error_trigger",
    "elevated_all_conditions",
]].round(3)

,participant_id,evaluations,eligible_evaluations,retained_while_waiting,mean_records_in_window,eligible_percent
0,F2-P002,9,9,0,3.890,100.000
1,F2-P003,9,4,5,2.440,44.440
2,P001,9,9,0,5.440,100.000


selected_state,participant_id,baseline,elevated,reduced
0,F2-P002,2,2,5
1,F2-P003,3,1,5
2,P001,1,4,4


,participant_id,record_count,il_ratio,fped_ratio,first_pass_error_rate,previous_state,selected_state,reason,reduced_il_trigger,reduced_fped_trigger,reduced_error_trigger,elevated_all_conditions
3,F2-P003,3,1.776,0.781,0.000,baseline,reduced,Recent performance met a reduced-state trigger.,True,False,False,False
4,F2-P003,3,1.320,0.733,0.000,reduced,reduced,Recent performance met a reduced-state trigger.,True,False,False,False
6,F2-P003,3,0.699,0.727,0.000,reduced,elevated,Recent performance met every elevated-state tr...,False,False,False,True
7,F2-P003,3,1.574,0.727,0.000,elevated,reduced,Recent performance met a reduced-state trigger.,True,False,False,False
9,P001,5,0.934,0.891,0.200,baseline,reduced,Recent performance met a reduced-state trigger.,False,False,True,False
10,P001,6,1.012,0.804,0.167,reduced,reduced,Recent performance met a reduced-state trigger.,False,False,True,False
11,P001,6,0.944,0.749,0.000,reduced,elevated,Recent performance met every elevated-state tr...,False,False,False,True
12,P001,6,0.899,0.775,0.000,elevated,elevated,Recent performance met every elevated-state tr...,False,False,False,True
13,P001,5,1.009,0.936,0.000,elevated,baseline,Neither reduced nor elevated trigger condition...,False,False,False,False
14,P001,5,0.865,0.940,0.200,baseline,reduced,Recent performance met a reduced-state trigger.,False,False,True,False


## 4. Music transitions

Direct Reduced ↔ Elevated changes are useful evidence when deciding whether to test a one-step policy through Baseline.

In [6]:
transitions["transition_duration_ms"] = (
    pd.to_numeric(transitions.completed_ms, errors="coerce")
    - pd.to_numeric(transitions.started_ms, errors="coerce")
)
transition_summary = transitions.groupby("participant_id", as_index=False).agg(
    transitions=("outcome", "size"),
    successful_playback=("outcome", lambda values: (values == "playing").sum()),
    mean_crossfade_ms=("transition_duration_ms", "mean"),
    min_crossfade_ms=("transition_duration_ms", "min"),
    max_crossfade_ms=("transition_duration_ms", "max"),
)
display(transition_summary.round(1))

direct_extreme_change = (
    ((decisions.previous_state == "reduced") & (decisions.selected_state == "elevated"))
    | ((decisions.previous_state == "elevated") & (decisions.selected_state == "reduced"))
)
direct_changes = decisions.loc[direct_extreme_change, [
    "participant_id", "previous_state", "selected_state", "record_count",
    "first_pass_error_rate", "reason",
]]
print(f"Direct Reduced ↔ Elevated changes: {len(direct_changes)}")
display(direct_changes)

print("Key pooled values")
print(f"Eligible decisions: {int(decisions.eligible.sum())} of {len(decisions)}")
print("Selected states:", decisions.selected_state.value_counts().to_dict())
print(f"Successful transitions: {(transitions.outcome == 'playing').sum()} of {len(transitions)}")

,participant_id,transitions,successful_playback,mean_crossfade_ms,min_crossfade_ms,max_crossfade_ms
0,F2-P002,5,5,"5,016.000","5,011.900","5,022.700"
1,F2-P003,3,3,"5,019.100","5,014.800","5,027.300"
2,P001,5,5,"5,137.800","5,093.000","5,173.000"


Direct Reduced ↔ Elevated changes: 6


,participant_id,previous_state,selected_state,record_count,first_pass_error_rate,reason
6,F2-P003,reduced,elevated,3,0.000,Recent performance met every elevated-state tr...
7,F2-P003,elevated,reduced,3,0.000,Recent performance met a reduced-state trigger.
11,P001,reduced,elevated,6,0.000,Recent performance met every elevated-state tr...
16,P001,reduced,elevated,5,0.000,Recent performance met every elevated-state tr...
19,F2-P002,elevated,reduced,4,0.250,Recent performance met a reduced-state trigger.
22,F2-P002,reduced,elevated,4,0.000,Recent performance met every elevated-state tr...


Key pooled values
Eligible decisions: 22 of 27
Selected states: {'reduced': 14, 'elevated': 7, 'baseline': 6}
Successful transitions: 13 of 13


## 5. Interpretation checklist

- Treat this as formative evidence, not a causal comparison of music conditions.
- Investigate unusually low Practice completion before drawing conclusions about Practice effectiveness.
- For low-throughput participants, test a longer rolling window before reducing the minimum-record count.
- If direct Reduced ↔ Elevated changes align with reports of noticeable transitions, test a one-step state policy through Baseline.
- Check build version, timing, and randomization seed before pooling later sessions.